<h1>Persiapan Awal (Konfigurasi & Setup Database)

In [1]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Overwriting konfigurasi.py


In [2]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK (jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print("\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

Writing setup_db_pengeluaran.py


<h1>Modul Akses Database (database.py)

In [3]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection() -> sqlite3.Connection | None:
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e: 
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        # Modifikasi untuk penugasan: kembalikan True jika berhasil (bukan hanya last_id dari INSERT)
        return last_id if last_id else True 
    except sqlite3.Error as e: 
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e: 
        print(f"ERROR [database.py] Fetch gagal: {e}")
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try: 
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e: 
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT, deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK (jumlah > 0), kategori TEXT, tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        return True
    except sqlite3.Error: return False
    finally:
        if conn: conn.close()

Writing database.py


<h1>Modul Model Data (model.py)

In [4]:
%%writefile model.py
import datetime
import locale

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal, id_transaksi: int = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")
            
        self.kategori = str(kategori) if kategori else "Lainnya"
        
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tgl '{tanggal}' salah.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tgl '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        try:
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        return f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, Jml:{jml_str}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')"

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "deskripsi": self.deskripsi, 
            "jumlah": self.jumlah,
            "kategori": self.kategori, 
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Writing model.py


<h1>Modul Manajer Anggaran (manajer_anggaran.py)

In [5]:
%%writefile manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0: return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None: 
            transaksi.id = last_id
            return True
        return False

    # === JAWABAN PENUGASAN: Menambahkan Fungsi Hapus ===
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        """Menghapus transaksi berdasarkan ID."""
        sql = "DELETE FROM transaksi WHERE id = ?"
        params = (id_transaksi,)
        hasil = database.execute_query(sql, params)
        return True if hasil else False
    # ====================================================

    def get_semua_transaksi_obj(self) -> list[Transaksi]:
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = database.fetch_query(sql, fetch_all=True)
        return [Transaksi(id_transaksi=r['id'], deskripsi=r['deskripsi'], jumlah=r['jumlah'], kategori=r['kategori'], tanggal=r['tanggal']) for r in rows] if rows else []

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi" # Modifikasi penugasan: menyertakan kolom 'id'
        params = None
        if filter_tanggal: 
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"
        df = database.get_dataframe(query, params=params)
        if not df.empty:
            try:
                import locale
                locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: locale.currency(x or 0, grouping=True, symbol='Rp ')[:-3])
            except: 
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']] # Menyertakan 'id' di tabel UI
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal: 
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        return float(result[0]) if result and result[0] is not None else 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal: 
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                hasil[row['kategori'] if row['kategori'] else "Lainnya"] = float(row[1]) if row[1] is not None else 0.0
        return hasil

Writing manajer_anggaran.py


<h1>Aplikasi Utama Streamlit (main_app.py)

In [6]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd
import locale

try: locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
except: pass

def format_rp(angka):
    try: return locale.currency(angka or 0, grouping=True, symbol='Rp ')[:-3]
    except: return f"Rp {angka or 0:,.0f}".replace(",",".")

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}")
    st.stop()

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide", initial_sidebar_state="expanded")

@st.cache_resource
def get_anggaran_manager(): return AnggaranHarian()
anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)
        
        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)*:", min_value=0.01, step=1000.0, format="%.0f", value=None)
        with col4: tanggal = st.date_input("Tanggal*:", value=datetime.date.today())
        
        if st.form_submit_button("Simpan Transaksi"):
            if not deskripsi or jumlah is None or jumlah <= 0:
                st.warning("Pastikan deskripsi dan jumlah terisi dengan benar!", icon="⚠️")
            else:
                with st.spinner("Menyimpan..."):
                    tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                    if anggaran.tambah_transaksi(tx):
                        st.success("OK! Simpan.", icon="✅")
                        st.cache_data.clear()
                        st.rerun()
                    else: st.error("Gagal simpan.", icon="❌")

def halaman_riwayat(anggaran: AnggaranHarian):
    st.subheader("Detail Semua Transaksi")
    if st.button("Refresh Riwayat"): 
        st.cache_data.clear()
        st.rerun()
        
    with st.spinner("Memuat riwayat..."): 
        df_transaksi = anggaran.get_dataframe_transaksi()
        
    if df_transaksi is None: st.error("Gagal ambil riwayat.")
    elif df_transaksi.empty: st.info("Belum ada transaksi.")
    else: 
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)
        
        # === JAWABAN PENUGASAN: UI Hapus Transaksi ===
        st.divider()
        st.subheader("Hapus Transaksi")
        st.write("Masukkan ID Transaksi yang ingin dihapus (lihat kolom 'id' pada tabel di atas).")
        with st.form("form_hapus_transaksi"):
            id_hapus = st.number_input("ID Transaksi:", min_value=1, step=1)
            if st.form_submit_button("Hapus Transaksi Terpilih"):
                if anggaran.hapus_transaksi(int(id_hapus)):
                    st.success(f"Transaksi dengan ID {id_hapus} berhasil dihapus.")
                    st.cache_data.clear()
                    st.rerun()
                else:
                    st.error(f"Gagal menghapus transaksi. ID {id_hapus} mungkin tidak ada.")
        # ===============================================

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.subheader("Ringkasan Pengeluaran")
    col_filter1, col_filter2 = st.columns([1, 2])
    with col_filter1:
        pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini", "Pilih Tanggal Tertentu"], on_change=lambda: st.cache_data.clear())
        tanggal_filter = datetime.date.today() if pilihan_periode == "Hari Ini" else None
        if pilihan_periode == "Pilih Tanggal Tertentu":
            tanggal_filter = st.date_input("Pilih Tanggal:", value=datetime.date.today(), on_change=lambda: st.cache_data.clear())
        label_periode = "(Semua Waktu)" if not tanggal_filter else f"({tanggal_filter.strftime('%d %b %Y')})"

    @st.cache_data(ttl=300)
    def hitung_total_cached(tgl): return anggaran.hitung_total_pengeluaran(tanggal=tgl)
    @st.cache_data(ttl=300)
    def get_kategori_cached(tgl): return anggaran.get_pengeluaran_per_kategori(tanggal=tgl)

    with col_filter2:
        st.metric(label=f"Total Pengeluaran {label_periode}", value=format_rp(hitung_total_cached(tanggal_filter)))
        
    st.divider()
    st.subheader(f"Pengeluaran per Kategori {label_periode}")
    
    with st.spinner("Memuat ringkasan kategori..."):
        dict_kat = get_kategori_cached(tanggal_filter)
        if not dict_kat: st.info("Tidak ada data untuk periode ini.")
        else:
            try:
                df_kat = pd.DataFrame([{"Kategori": k, "Total": v} for k, v in dict_kat.items()]).sort_values(by="Total", ascending=False).reset_index(drop=True)
                df_kat['Total (Rp)'] = df_kat['Total'].apply(format_rp)
                c1, c2 = st.columns(2)
                with c1: st.dataframe(df_kat[['Kategori', 'Total (Rp)']], hide_index=True, use_container_width=True)
                with c2: st.bar_chart(df_kat.set_index('Kategori')['Total'], use_container_width=True)
            except Exception as e: st.error(f"Gagal tampilkan ringkasan: {e}")

def main():
    st.sidebar.title("Catatan Pengeluaran")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
    st.sidebar.markdown("---")
    st.sidebar.info("Jobsheet - Aplikasi Keuangan")
    
    if menu_pilihan == "Tambah": halaman_input(anggaran)
    elif menu_pilihan == "Riwayat": halaman_riwayat(anggaran)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(anggaran)
        
    st.markdown("---")
    st.caption("Pengembangan Aplikasi Berbasis OOP")

if __name__ == "__main__":
    main()

Writing main_app.py
